In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.optim as optim
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from torchvision.transforms import v2

In [ ]:
train_dataset = datasets.FashionMNIST(
    root='./fmnist',
    train=True,
    download=True,
)

test_dataset = datasets.FashionMNIST(
    root='./fmnist',
    train=False,
    download=True,
)

In [ ]:
print(len(train_dataset))
print(len(test_dataset))

In [ ]:
def visualize_data(image, label):
    classes = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]
    plt.figure(figsize=(1,1))
    plt.imshow(image, cmap='gray')
    plt.xlabel(classes[label])
    plt.show()
for i in range(3):
    image = train_dataset[i][0]
    label = train_dataset[i][1]
    visualize_data(image, label)

## 기본 전처리 + 데이터

In [ ]:
transforms = v2.Compose(
    [
        v2.ToImage(),
        v2.ToDtype(dtype=torch.float32, scale=True)
    ]
)

In [ ]:
train_dataset = datasets.FashionMNIST(
    root='./fmnist',
    train=True,
    download=True,
    transform=transforms,
)

test_dataset = datasets.FashionMNIST(
    root='./fmnist',
    train=False,
    download=True,
    transform=transforms,
)

In [ ]:
train_dataset[0][0].shape

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# 모델 생성 및 학습


## 모델 생성

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class BasicAutoEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(28*28, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 28*28),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

In [ ]:
model = BasicAutoEncoder().to(device)

loss_fn = nn.MSELoss()
opt = optim.Adam(model.parameters(), lr = 0.0001)

In [ ]:
# 학습 루프
# 베이직 인코더 1번
epochs = 10
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch in train_loader:
        images = batch[0].to(device)
        images = images.view(images.size(0), -1)
        outputs = model(images)
        loss = loss_fn(outputs, images)

        loss.backward()
        opt.step()
        opt.zero_grad()

        total_loss += loss.item()
    print(f'Epoch : {epoch+1}/{epochs}, Loss : {total_loss/len(train_loader)}')


In [ ]:
# test
# 베이직 인코더 1번
model.eval()
with torch.no_grad():
    for images, _ in test_loader:
        images = images.view(images.size(0), -1).to(device)
        outputs = model(images)
        break

images = images.view(-1, 28,28)
outputs = outputs.view(-1, 28,28)

fig, axes = plt.subplots(2,10, figsize=(15,6))

for i in range(10):
    axes[0,i].imshow(images[i].cpu().numpy(), cmap='gray')
    axes[0,i].set_title(f"Origin {i+1}")
    axes[0,i].axis("off")

    axes[1,i].imshow(outputs[i].cpu().numpy(), cmap='gray')
    axes[1,i].set_title(f"Recon {i+1}")
    axes[1,i].axis("off")

## 3D View

In [ ]:
class BasicAutoEncoder2(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(28*28, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 3),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(3, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 28*28),
            nn.Sigmoid()
        )
    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

In [ ]:
model2 = BasicAutoEncoder2().to(device)

loss_fn = nn.MSELoss()
opt2 = optim.Adam(model2.parameters(), lr = 0.0001)

In [ ]:
# 학습 루프
#베이직 인코더 2번

epochs = 10
for epoch in range(epochs):
    model2.train()
    total_loss = 0
    for batch in train_loader:
        images = batch[0].to(device)
        images = images.view(images.size(0), -1)
        outputs = model2(images)
        loss = loss_fn(outputs, images)

        loss.backward()
        opt2.step()
        opt2.zero_grad()

        total_loss += loss.item()
    print(f'Epoch : {epoch+1}/{epochs}, Loss : {total_loss/len(train_loader)}')

In [ ]:
# 테스트
# 베이직 인코더 2번
model2.eval()
with torch.no_grad():
    for images, _ in test_loader:
        images = images.view(images.size(0), -1).to(device)
        outputs = model2(images)
        break

images = images.view(-1, 28,28)
outputs = outputs.view(-1, 28,28)

fig, axes = plt.subplots(2,10, figsize=(15,6))

for i in range(10):
    axes[0,i].imshow(images[i].cpu().numpy(), cmap='gray')
    axes[0,i].set_title(f"Origin {i+1}")
    axes[0,i].axis("off")

    axes[1,i].imshow(outputs[i].cpu().numpy(), cmap='gray')
    axes[1,i].set_title(f"Recon {i+1}")
    axes[1,i].axis("off")

In [ ]:
# latent 공간 시각화를 위한 데이터 추출
model2.eval()
all_latents = []
all_labels = []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.view(images.size(0),-1).to(device)
        latent = model2.encoder(images)
        all_latents.append(latent.cpu().numpy())
        all_labels.append(labels.cpu().numpy())


In [ ]:
all_latents = np.concatenate(all_latents, axis=0)
all_labels = np.concatenate(all_labels, axis=0)

#3D 시각화 : 전체 분화 일부 샘플에 텍스트 어노테이션 추가
fig = plt.figure(figsize=(12,10))
ax = fig.add_subplot(111,projection='3d')

# 전체 latent 벡터를 scatter plot
scatter = ax.scatter(all_latents[:,0], all_latents[:,1], all_latents[:,2],
                     c=all_labels, cmap='tab10', s=10, alpha=0.3)

# 각 digit별로 일부 포인트에 텍스트 어노테이션 추가
for digit in range(10):
    indices = np.where(all_labels == digit)[0]
    sample_indices = np.random.choice(indices, size=min(50, len(indices)), replace=False)

    for idx in sample_indices:
        x,y,z = all_latents[idx, 0], all_latents[idx, 1], all_latents[idx, 2]
        ax.text(x,y,z, str(digit),
                color=plt.cm.tab10(digit),
                fontsize=10,
                fontweight='bold',
                ha='center', va='center')

ax.set_xlabel('dim 1')
ax.set_ylabel('dim 2')
ax.set_zlabel('dim 3')
plt.show()

In [ ]:
## 다른방법
import plotly.graph_objects as go
import numpy as np

# 각 숫자(0~9)에 대한 색상 리스트 (Plotly 기본 색상 사용)
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
          "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf"]

fig = go.Figure()

for digit in np.unique(all_labels):
    indices = np.where(all_labels == digit)[0]
    sample_indices = np.random.choice(indices, size=min(50, len(indices)), replace=False)

    fig.add_trace(go.Scatter3d(
        x=all_latents[sample_indices,0],
        y=all_latents[sample_indices,1],
        z=all_latents[sample_indices,2],
        mode='text',
        text=[str(digit)] * len(sample_indices),
        textposition='middle center',
        name = f"Digit {digit}"
    ))

fig.update_layout(
    scene=dict(
        xaxis_title = 'dim 1',
        yaxis_title = 'dim 2',
        zaxis_title = 'dim 3',
    ),
    width = 800,
    margin=dict(r=20,l=10,b=10,t=10)
)

fig.show()


# CNN Autoencoder

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [ ]:
# 2. CNN 모델 생성
class CNNAutoEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        #Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, 1, 1),
            nn.ReLU(True),
            nn.MaxPool2d(2,2),
            nn.Conv2d(16, 32, 3, 1, 1),
            nn.ReLU(True),
            nn.MaxPool2d(2,2)
        )
        #Decoder
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2),
            nn.ReLU(True),
            nn.ConvTranspose2d(16, 1, kernel_size=2, stride=2),
            nn.Sigmoid()
        )
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded, encoded

In [ ]:
# 모델 학습
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CNNAutoEncoder().to(device)
loss_fn = nn.MSELoss()
opt = optim.Adam(model.parameters(), lr = 0.001)

epochs = 5
num_ch = 5
for epoch in range(epochs):
    model.train()
    running_loss = 0
    for data in train_loader:
        inputs = data[0].to(device)
        outputs, encoded = model(inputs)

        loss = loss_fn(outputs, inputs)
        loss.backward()
        opt.step()
        opt.zero_grad()

        running_loss += loss.item()

    for c in range(num_ch):
        ax = plt.subplot(num_ch, epochs, epoch+c * epochs + 1)
        encoded_img_channel = encoded[epoch, c, :,:].detach().cpu().numpy()
        plt.imshow(encoded_img_channel, cmap='viridis')
        ax.get_xaxis().set_visible(False)
        ax.get_yaxis().set_visible(False)

In [ ]:
model.eval()
with torch.no_grad():
    dataiter = iter(test_loader)
    images, labels = next(dataiter)
    outputs, encoded = model(images)

    n = 10
    num_ch = 6

    plt.figure(figsize=(20,10))
    for i in range(n):
        ax = plt.subplot(num_ch+2, n, i+1)
        img = images[i].cpu().numpy()
        plt.imshow(img.squeeze(), cmap='gray')
        if i==0:
            ax.set_title("Origin")

        for c in range(num_ch):
            ax = plt.subplot(num_ch+2, n, n+i+c*n+1)
            encoded_img_channel = encoded[i, c, :,:].cpu().numpy()
            plt.imshow(encoded_img_channel, cmap='viridis')

        ax = plt.subplot(num_ch+2, n, (num_ch+1)*n+i+1)
        decoded_img = outputs[i].cpu().numpy().squeeze()
        plt.imshow(decoded_img, cmap='gray')
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
# Function to generate a new image from a random latent vector
def generate_image(model):
    sample_num = 10
    model.eval()  # Set the model to evaluation mode
    with torch.no_grad():
        random_latent_vector = encoded[sample_num].unsqueeze(0) * 0.2 + 0.2
        generated_image = model.decoder(random_latent_vector)
        generated_img_np = generated_image.squeeze().cpu().numpy()

        plt.figure(figsize=(5, 5))
        plt.subplot(2,1,1)
        plt.imshow(images[sample_num].squeeze(), cmap='gray')
        plt.subplot(2,1,2)
        plt.imshow(generated_img_np, cmap='gray')
        plt.title("Generated Image")
        plt.axis('off')
        plt.show()

# Generate a new image using the trained model
generate_image(model)